# Prédiction Match Depuis Les Clusters Joueurs

Objectif : prédire le résultat d'un match (`1`, `0`, `-1`) à partir des profils de clusters des 22 titulaires.

La version actuelle utilise uniquement les `cluster_key` des joueurs, jamais les `cluster_id` seuls.

## TL;DR De La Dernière Validation

- 3 473 matchs éligibles en analyse globale.
- 1 004 matchs éligibles en analyse `same_rank_bucket`.
- 29 `cluster_key` distincts validés, avec `cluster_key` utilisé comme feature, jamais `cluster_id` seul.
- 98 équipes Neon sur 99 mappées au ranking top 100 ; seule `Paris FC` reste hors ranking.
- Meilleur modèle global : `random_forest`, balanced accuracy `0.406`, macro F1 `0.404`.
- Meilleur modèle same bucket : `logistic_regression`, balanced accuracy `0.403`, macro F1 `0.389`.

Les scores sont modestes, ce qui est normal pour une première version qui n'utilise que les profils des titulaires, sans niveau absolu, forme récente, domicile/extérieur ou dynamique d'équipe.


## 1. Setup

Paramètres, imports et chemins utilisés dans le notebook.

In [ ]:
from pathlib import Path
import os
import re
import unicodedata

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sqlalchemy import create_engine
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix, f1_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

pd.set_option("display.max_columns", 160)
pd.set_option("display.max_rows", 120)

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SCHEMA = "public"
RANDOM_STATE = 42
TEST_SIZE = 0.20
RANKING_YEAR = 2026
TARGET_LABELS = [-1, 0, 1]

CLUSTERS_PATH = PROJECT_ROOT / "analysis_outputs" / "player_role_clusters_with_style_df.csv"
RANKING_PATH = Path(
    "/Users/hugopierre/projets/formation jedha/foot-predictor/data/raw/club_clustering/uefa_top_100_clubs.csv"
)
OUTPUT_DIR = PROJECT_ROOT / "analysis_outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

POSITION_ORDER = {
    "Goalkeeper": 1,
    "Centre-Back": 2,
    "Left-Back": 3,
    "Right-Back": 4,
    "Sweeper": 5,
    "Defensive Midfield": 6,
    "Central Midfield": 7,
    "Attacking Midfield": 8,
    "Left Midfield": 9,
    "Right Midfield": 10,
    "Left Winger": 11,
    "Right Winger": 12,
    "Second Striker": 13,
    "Centre-Forward": 14,
}

print("Projet:", PROJECT_ROOT)
print("Clusters:", CLUSTERS_PATH)
print("Ranking:", RANKING_PATH)

## 2. Connexion Neon

La connexion lit `NEON_DATABASE_URL`, `DATABASE_URL` ou `POSTGRES_URL` depuis l'environnement ou le `.env`.

In [ ]:
def load_env_file(path: Path) -> dict[str, str]:
    values = {}

    if not path.exists():
        return values

    for raw_line in path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()

        if not line or line.startswith("#") or "=" not in line:
            continue

        key, value = line.split("=", 1)
        values[key.strip()] = value.strip().strip('"').strip("'")

    return values


def sqlalchemy_url(url: str) -> str:
    if url.startswith("postgres://"):
        return url.replace("postgres://", "postgresql+psycopg2://", 1)

    if url.startswith("postgresql://"):
        return url.replace("postgresql://", "postgresql+psycopg2://", 1)

    return url


env_path = PROJECT_ROOT / ".env"
local_env = load_env_file(env_path)
DATABASE_URL = (
    os.environ.get("NEON_DATABASE_URL")
    or os.environ.get("DATABASE_URL")
    or os.environ.get("POSTGRES_URL")
    or local_env.get("NEON_DATABASE_URL")
    or local_env.get("DATABASE_URL")
    or local_env.get("POSTGRES_URL")
    or ""
)

if not DATABASE_URL:
    raise RuntimeError("Ajoute NEON_DATABASE_URL, DATABASE_URL ou POSTGRES_URL dans .env.")

engine = create_engine(sqlalchemy_url(DATABASE_URL), pool_pre_ping=True)
print("Connexion configurée")

## 3. Charger Les Données

On charge seulement les tables nécessaires : matchs, équipes par match, lineups et référentiel équipes.

In [ ]:
matches_df = pd.read_sql(
    f'''
    SELECT match_id, match_date, competition_id, round, stadium, referee
    FROM "{SCHEMA}"."match"
    ''',
    engine,
)

match_team_df = pd.read_sql(
    f'''
    SELECT match_id, team_id, side, score, penalty_score, manager_name, formation
    FROM "{SCHEMA}"."match_team"
    ''',
    engine,
)

lineup_df = pd.read_sql(
    f'''
    SELECT match_id, team_id, player_id, position_player, is_starting_match
    FROM "{SCHEMA}"."lineup"
    ''',
    engine,
)

teams_df = pd.read_sql(
    f'''
    SELECT team_id, team_name
    FROM "{SCHEMA}"."team"
    ''',
    engine,
)

clusters_df = pd.read_csv(CLUSTERS_PATH)
ranking_raw_df = pd.read_csv(RANKING_PATH)

print("match", matches_df.shape)
print("match_team", match_team_df.shape)
print("lineup", lineup_df.shape)
print("team", teams_df.shape)
print("clusters", clusters_df.shape)
print("ranking", ranking_raw_df.shape)

display(matches_df.head())
display(match_team_df.head())
display(lineup_df.head())
display(clusters_df.head())

## 4. Contrôler Les Clusters Joueurs

On vérifie que la feature ML est bien `cluster_key`, unique par rôle + cluster, et jamais `cluster_id` seul.

In [ ]:
required_cluster_columns = [
    "player_id", "best_position", "position_role", "cluster_id", "cluster_key", "style_label"
]
missing_cluster_columns = [column for column in required_cluster_columns if column not in clusters_df.columns]
if missing_cluster_columns:
    raise KeyError(f"Colonnes manquantes dans le CSV clusters: {missing_cluster_columns}")

clusters_df = clusters_df[required_cluster_columns].copy()
clusters_df["player_id"] = pd.to_numeric(clusters_df["player_id"], errors="raise").astype(int)
clusters_df["cluster_id"] = pd.to_numeric(clusters_df["cluster_id"], errors="raise").astype(int)
clusters_df["expected_cluster_key"] = clusters_df["position_role"] + "_" + clusters_df["cluster_id"].astype(str)

bad_cluster_key_df = clusters_df[clusters_df["cluster_key"] != clusters_df["expected_cluster_key"]]
if not bad_cluster_key_df.empty:
    raise ValueError("Certains cluster_key ne correspondent pas à position_role + cluster_id.")

cluster_key_control_df = (
    clusters_df
    .groupby("cluster_key")
    .agg(
        position_role_count=("position_role", "nunique"),
        cluster_id_count=("cluster_id", "nunique"),
        player_count=("player_id", "count"),
        style_label=("style_label", "first"),
    )
    .reset_index()
    .sort_values("cluster_key")
)

if cluster_key_control_df["position_role_count"].max() != 1:
    raise ValueError("Un cluster_key correspond à plusieurs rôles.")

if cluster_key_control_df["cluster_id_count"].max() != 1:
    raise ValueError("Un cluster_key correspond à plusieurs cluster_id.")

if cluster_key_control_df["cluster_key"].nunique() != 29:
    raise ValueError("On attend exactement 29 cluster_key distincts.")

clusters_df = clusters_df.drop(columns="expected_cluster_key")

print("Cluster_key OK:", cluster_key_control_df["cluster_key"].nunique(), "clusters distincts")
display(cluster_key_control_df)

## 5. Aperçu Joueur -> Cluster

Petit aperçu de 5 joueurs avec le `cluster_key` correspondant.


In [ ]:
players_head_df = pd.read_sql(
    f'''
    SELECT player_id, player_name, best_position
    FROM "{SCHEMA}"."player"
    ORDER BY player_name
    LIMIT 100
    ''',
    engine,
)

players_head_df["player_id"] = pd.to_numeric(players_head_df["player_id"], errors="raise").astype(int)

players_head_df = (
    players_head_df
    .merge(clusters_df[["player_id", "cluster_key"]], on="player_id", how="inner")
    [["player_id", "player_name", "best_position", "cluster_key"]]
    .head(5)
)

display(players_head_df)


## 6. Préparer Le Ranking Top 100

Le ranking vient du CSV existant `uefa_top_100_clubs.csv`. Il sert uniquement à filtrer les matchs où les deux équipes sont dans le même bucket de 20 places.

In [ ]:
def normalize_team_name(value):
    if pd.isna(value):
        return ""

    text = str(value).replace("ß", "ss")
    text = unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode("ascii")
    text = text.lower().replace("&", " and ")
    text = re.sub(r"[^a-z0-9]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


TEAM_NAME_TO_RANKING_KEY_OVERRIDES = {
    "1 fc union berlin": "union berlin",
    "ac milan": "milan",
    "ac sparta prague": "sparta praha",
    "acf fiorentina": "fiorentina",
    "as monaco": "monaco",
    "arsenal fc": "arsenal",
    "associazione sportiva roma": "roma",
    "atalanta bc": "atalanta",
    "atletico de madrid": "atletico madrid",
    "bsc young boys": "young boys",
    "bayer 04 leverkusen": "bayer leverkusen",
    "bayern munich": "bayern munchen",
    "bologna football club 1909": "bologna",
    "celtic fc": "celtic",
    "chelsea fc": "chelsea",
    "club brugge kv": "club brugge",
    "djurgardens idrottsforening": "djurgardens if",
    "fc barcelona": "barcelona",
    "fc basel 1893": "basel",
    "fc copenhagen": "copenhagen",
    "fc lugano": "lugano",
    "fc midtjylland": "midtjylland",
    "fc porto": "porto",
    "fc shakhtar donetsk": "shakhtar donetsk",
    "fc viktoria plzen": "viktoria plzen",
    "fk partizan belgrade": "partizan",
    "ferencvarosi tc": "ferencvaros",
    "feyenoord rotterdam": "feyenoord",
    "fotballklubben bodo glimt": "bodo glimt",
    "fotballklubben bod glimt": "bodo glimt",
    "fussballclub red bull salzburg": "salzburg",
    "gnk dinamo zagreb": "dinamo zagreb",
    "inter milan": "inter",
    "juventus fc": "juventus",
    "kaa gent": "gent",
    "krc genk": "genk",
    "losc lille": "lille osc",
    "legia warszawa": "legia warsaw",
    "liverpool fc": "liverpool",
    "molde fotballklubb": "molde",
    "nk celje": "celje",
    "olympiakos syndesmos filathlon peiraios": "olympiakos",
    "olympique lyon": "lyon",
    "olympique marseille": "marseille",
    "pafos fc": "pafos",
    "panathinaikos athlitikos omilos": "panathinaikos",
    "panthessalonikios athlitikos omilos konstantinoupoliton": "paok thessaloniki",
    "qarabag fk": "qarabag",
    "rc strasbourg alsace": "strasbourg",
    "rsc anderlecht": "anderlecht",
    "rangers fc": "rangers",
    "rapid vienna": "rapid wien",
    "real betis balompie": "real betis",
    "sc braga": "sporting braga",
    "sc freiburg": "freiburg",
    "sk slavia prague": "slavia prague",
    "sl benfica": "benfica",
    "ssc napoli": "napoli",
    "sevilla fc": "sevilla",
    "societa sportiva lazio s p a": "lazio",
    "sporting cp": "sporting cp lisbon",
    "sportklub puntigamer sturm graz": "sturm graz",
    "stade rennais fc": "stade rennais",
    "vfb stuttgart": "stuttgart",
    "villarreal cf": "villarreal",
}

ranking_df = ranking_raw_df[ranking_raw_df["ranking_year"].eq(RANKING_YEAR)].copy()
ranking_df["ranking_key"] = ranking_df["club_key"].map(normalize_team_name)
ranking_df["rank_bucket"] = ((ranking_df["rank"] - 1) // 20 + 1).astype(int)

if ranking_df["ranking_key"].duplicated().any():
    duplicated_keys = ranking_df[ranking_df["ranking_key"].duplicated(keep=False)]
    raise ValueError(f"ranking_key dupliqués dans le ranking: {duplicated_keys[['club_name', 'ranking_key']]}")

ranking_lookup_df = ranking_df[[
    "ranking_year", "rank", "ranking_row", "club_name", "country_code",
    "total_points", "club_key", "ranking_key", "rank_bucket",
]].copy()

teams_rank_df = teams_df.copy()
teams_rank_df["team_normalized"] = teams_rank_df["team_name"].map(normalize_team_name)
teams_rank_df["ranking_key"] = teams_rank_df["team_normalized"].map(
    lambda value: TEAM_NAME_TO_RANKING_KEY_OVERRIDES.get(value, value)
)

team_rank_mapping_df = teams_rank_df.merge(
    ranking_lookup_df,
    on="ranking_key",
    how="left",
)

unmapped_teams_df = (
    team_rank_mapping_df[team_rank_mapping_df["rank"].isna()]
    [["team_id", "team_name", "team_normalized", "ranking_key"]]
    .sort_values("team_name")
    .reset_index(drop=True)
)

print("Équipes mappées ranking:", team_rank_mapping_df["rank"].notna().sum(), "/", len(team_rank_mapping_df))
print("Équipes non mappées:", len(unmapped_teams_df))
display(team_rank_mapping_df.sort_values("rank").head(20))
display(unmapped_teams_df)

## 7. Construire Les 22 Features Match

Une ligne = un match. Les 11 titulaires de chaque équipe deviennent des colonnes catégorielles `cluster_key`.

In [ ]:
home_team_df = (
    match_team_df[match_team_df["side"].eq("home")]
    .rename(columns={"team_id": "home_team_id", "score": "home_score"})
    [["match_id", "home_team_id", "home_score", "manager_name", "formation"]]
    .rename(columns={"manager_name": "home_manager_name", "formation": "home_formation"})
)

away_team_df = (
    match_team_df[match_team_df["side"].eq("away")]
    .rename(columns={"team_id": "away_team_id", "score": "away_score"})
    [["match_id", "away_team_id", "away_score", "manager_name", "formation"]]
    .rename(columns={"manager_name": "away_manager_name", "formation": "away_formation"})
)

base_match_df = (
    matches_df
    .merge(home_team_df, on="match_id", how="inner")
    .merge(away_team_df, on="match_id", how="inner")
    .merge(teams_df.rename(columns={"team_id": "home_team_id", "team_name": "home_team_name"}), on="home_team_id", how="left")
    .merge(teams_df.rename(columns={"team_id": "away_team_id", "team_name": "away_team_name"}), on="away_team_id", how="left")
)

base_match_df["home_score"] = pd.to_numeric(base_match_df["home_score"], errors="coerce")
base_match_df["away_score"] = pd.to_numeric(base_match_df["away_score"], errors="coerce")
base_match_df = base_match_df[base_match_df["home_score"].notna() & base_match_df["away_score"].notna()].copy()
base_match_df["target_result"] = 0
base_match_df.loc[base_match_df["home_score"] > base_match_df["away_score"], "target_result"] = 1
base_match_df.loc[base_match_df["home_score"] < base_match_df["away_score"], "target_result"] = -1

lineup_side_df = lineup_df.merge(
    match_team_df[["match_id", "team_id", "side"]],
    on=["match_id", "team_id"],
    how="left",
)

starters_df = lineup_side_df[lineup_side_df["is_starting_match"].fillna(0).astype(int).eq(1)].copy()
starters_df["player_id"] = pd.to_numeric(starters_df["player_id"], errors="raise").astype(int)
starters_df = starters_df.merge(
    clusters_df[["player_id", "cluster_key", "style_label"]],
    on="player_id",
    how="left",
)
starters_df["is_goalkeeper"] = starters_df["position_player"].eq("Goalkeeper")
starters_df["position_order"] = starters_df["position_player"].map(POSITION_ORDER).fillna(99).astype(int)
starters_df["has_cluster"] = starters_df["cluster_key"].notna()

team_lineup_quality_df = (
    starters_df
    .groupby(["match_id", "team_id", "side"], dropna=False)
    .agg(
        starter_count=("player_id", "count"),
        clustered_starter_count=("has_cluster", "sum"),
        goalkeeper_count=("is_goalkeeper", "sum"),
    )
    .reset_index()
)

complete_team_lineups_df = team_lineup_quality_df[
    team_lineup_quality_df["starter_count"].eq(11)
    & team_lineup_quality_df["clustered_starter_count"].eq(11)
    & team_lineup_quality_df["goalkeeper_count"].eq(1)
].copy()

complete_starters_df = starters_df.merge(
    complete_team_lineups_df[["match_id", "team_id", "side"]],
    on=["match_id", "team_id", "side"],
    how="inner",
)

complete_starters_df = complete_starters_df.sort_values([
    "match_id", "team_id", "side", "position_order", "cluster_key", "player_id"
]).copy()
complete_starters_df["slot_number"] = (
    complete_starters_df
    .groupby(["match_id", "team_id", "side"])
    .cumcount()
    .add(1)
)

bad_slot_df = complete_starters_df[complete_starters_df["slot_number"].gt(11)]
if not bad_slot_df.empty:
    raise ValueError("Certaines équipes ont plus de 11 slots après filtrage.")

first_slot_check_df = complete_starters_df[complete_starters_df["slot_number"].eq(1)]
if not first_slot_check_df["position_player"].eq("Goalkeeper").all():
    raise ValueError("Le slot 1 doit toujours être le gardien.")

slot_cluster_df = (
    complete_starters_df
    .pivot_table(
        index=["match_id", "team_id", "side"],
        columns="slot_number",
        values="cluster_key",
        aggfunc="first",
    )
    .reset_index()
)
slot_cluster_df.columns = [
    str(column) if not isinstance(column, int) else f"player_{column:02d}_cluster_key"
    for column in slot_cluster_df.columns
]

home_slots_df = (
    slot_cluster_df[slot_cluster_df["side"].eq("home")]
    .drop(columns="side")
    .rename(columns={"team_id": "home_team_id"})
)
home_slots_df = home_slots_df.rename(columns={
    f"player_{slot:02d}_cluster_key": f"home_player_{slot:02d}_cluster_key"
    for slot in range(1, 12)
})

away_slots_df = (
    slot_cluster_df[slot_cluster_df["side"].eq("away")]
    .drop(columns="side")
    .rename(columns={"team_id": "away_team_id"})
)
away_slots_df = away_slots_df.rename(columns={
    f"player_{slot:02d}_cluster_key": f"away_player_{slot:02d}_cluster_key"
    for slot in range(1, 12)
})

home_feature_columns = [f"home_player_{slot:02d}_cluster_key" for slot in range(1, 12)]
away_feature_columns = [f"away_player_{slot:02d}_cluster_key" for slot in range(1, 12)]
feature_columns = home_feature_columns + away_feature_columns

match_cluster_dataset_all_df = (
    base_match_df
    .merge(home_slots_df, on=["match_id", "home_team_id"], how="inner")
    .merge(away_slots_df, on=["match_id", "away_team_id"], how="inner")
)

missing_feature_rows = match_cluster_dataset_all_df[feature_columns].isna().any(axis=1).sum()
if missing_feature_rows:
    raise ValueError(f"{missing_feature_rows} matchs ont des clusters manquants dans les 22 slots.")

print("Matchs avec score:", len(base_match_df))
print("Équipes avec 11 titulaires + 11 clusters + 1 gardien:", len(complete_team_lineups_df))
print("Matchs éligibles global:", len(match_cluster_dataset_all_df))
print("Distribution target")
display(match_cluster_dataset_all_df["target_result"].value_counts().sort_index())
display(match_cluster_dataset_all_df[["match_id", "match_date", "home_team_name", "away_team_name", "home_score", "away_score", "target_result"] + feature_columns].head())

## 8. Ajouter Les Buckets UEFA

Le dataset global conserve tous les matchs éligibles. Le dataset bucket conserve seulement les matchs où les deux équipes sont dans le même intervalle de ranking.

In [ ]:
team_rank_features_df = team_rank_mapping_df[[
    "team_id", "rank", "ranking_row", "club_name", "club_key", "total_points", "rank_bucket"
]].copy()

home_rank_df = team_rank_features_df.rename(columns={
    "team_id": "home_team_id",
    "rank": "home_rank",
    "ranking_row": "home_ranking_row",
    "club_name": "home_ranking_club_name",
    "club_key": "home_ranking_club_key",
    "total_points": "home_ranking_points",
    "rank_bucket": "home_rank_bucket",
})

away_rank_df = team_rank_features_df.rename(columns={
    "team_id": "away_team_id",
    "rank": "away_rank",
    "ranking_row": "away_ranking_row",
    "club_name": "away_ranking_club_name",
    "club_key": "away_ranking_club_key",
    "total_points": "away_ranking_points",
    "rank_bucket": "away_rank_bucket",
})

match_cluster_dataset_all_df = (
    match_cluster_dataset_all_df
    .merge(home_rank_df, on="home_team_id", how="left")
    .merge(away_rank_df, on="away_team_id", how="left")
)

same_bucket_mask = (
    match_cluster_dataset_all_df["home_rank_bucket"].notna()
    & match_cluster_dataset_all_df["away_rank_bucket"].notna()
    & (
        match_cluster_dataset_all_df["home_rank_bucket"].astype("Int64")
        == match_cluster_dataset_all_df["away_rank_bucket"].astype("Int64")
    )
)
match_cluster_dataset_same_rank_bucket_df = match_cluster_dataset_all_df[same_bucket_mask].copy()

print("Matchs éligibles global:", len(match_cluster_dataset_all_df))
print("Matchs même bucket UEFA:", len(match_cluster_dataset_same_rank_bucket_df))
print("Distribution bucket")
display(match_cluster_dataset_same_rank_bucket_df["home_rank_bucket"].value_counts().sort_index())
print("Distribution target global")
display(match_cluster_dataset_all_df["target_result"].value_counts().sort_index())
print("Distribution target même bucket")
display(match_cluster_dataset_same_rank_bucket_df["target_result"].value_counts().sort_index())

## 9. Sauvegarder Les Datasets

On exporte les datasets pour pouvoir les relire sans refaire les joins Neon.

In [ ]:
match_cluster_dataset_all_df.to_csv(OUTPUT_DIR / "match_cluster_dataset_all.csv", index=False)
match_cluster_dataset_same_rank_bucket_df.to_csv(
    OUTPUT_DIR / "match_cluster_dataset_same_rank_bucket.csv",
    index=False,
)
team_rank_mapping_df.to_csv(OUTPUT_DIR / "team_rank_mapping_df.csv", index=False)
unmapped_teams_df.to_csv(OUTPUT_DIR / "unmapped_ranking_teams_df.csv", index=False)
cluster_key_control_df.to_csv(OUTPUT_DIR / "match_cluster_key_control_df.csv", index=False)

print("Exports datasets écrits dans", OUTPUT_DIR)

## 10. Fonctions D'Évaluation ML

Même split et mêmes métriques pour les deux analyses.

In [ ]:
def chronological_train_test_split(frame, test_size=TEST_SIZE):
    ordered_df = frame.sort_values(["match_date", "match_id"]).reset_index(drop=True)
    split_index = int(len(ordered_df) * (1 - test_size))

    if split_index <= 0 or split_index >= len(ordered_df):
        raise ValueError(f"Dataset trop petit pour split train/test: {len(ordered_df)} lignes")

    train_df = ordered_df.iloc[:split_index].copy()
    test_df = ordered_df.iloc[split_index:].copy()
    return train_df, test_df


def get_models():
    return {
        "dummy_most_frequent": DummyClassifier(strategy="most_frequent"),
        "logistic_regression": Pipeline([
            ("one_hot", OneHotEncoder(handle_unknown="ignore")),
            ("model", LogisticRegression(
                max_iter=2000,
                class_weight="balanced",
                C=1.0,
                solver="lbfgs",
                random_state=RANDOM_STATE,
            )),
        ]),
        "random_forest": Pipeline([
            ("one_hot", OneHotEncoder(handle_unknown="ignore")),
            ("model", RandomForestClassifier(
                n_estimators=300,
                max_depth=8,
                min_samples_leaf=10,
                class_weight="balanced_subsample",
                random_state=RANDOM_STATE,
                n_jobs=-1,
            )),
        ]),
    }


def evaluate_dataset(dataset_name, dataset_df):
    train_df, test_df = chronological_train_test_split(dataset_df)
    X_train = train_df[feature_columns]
    y_train = train_df["target_result"]
    X_test = test_df[feature_columns]
    y_test = test_df["target_result"]

    rows = []
    confusion_rows = []

    print(f"\n{dataset_name}")
    print("train:", len(train_df), "test:", len(test_df))
    print("train target")
    display(y_train.value_counts().sort_index())
    print("test target")
    display(y_test.value_counts().sort_index())

    for model_name, model in get_models().items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        matrix = confusion_matrix(y_test, y_pred, labels=TARGET_LABELS)
        metrics = {
            "dataset": dataset_name,
            "model": model_name,
            "train_rows": len(train_df),
            "test_rows": len(test_df),
            "accuracy": accuracy_score(y_test, y_pred),
            "balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
            "macro_f1": f1_score(y_test, y_pred, average="macro", zero_division=0),
            "test_start_date": test_df["match_date"].min(),
            "test_end_date": test_df["match_date"].max(),
        }
        rows.append(metrics)

        for actual_index, actual_label in enumerate(TARGET_LABELS):
            for predicted_index, predicted_label in enumerate(TARGET_LABELS):
                confusion_rows.append({
                    "dataset": dataset_name,
                    "model": model_name,
                    "actual": actual_label,
                    "predicted": predicted_label,
                    "rows": int(matrix[actual_index, predicted_index]),
                })

        print(model_name)
        print(pd.Series(metrics).drop(["dataset", "model"]).to_string())
        display(pd.DataFrame(matrix, index=[f"actual_{label}" for label in TARGET_LABELS], columns=[f"pred_{label}" for label in TARGET_LABELS]))

    return pd.DataFrame(rows), pd.DataFrame(confusion_rows)

## 11. Analyse Globale

Premier modèle : tous les matchs éligibles, sans restriction de niveau UEFA.

In [ ]:
global_metrics_df, global_confusion_df = evaluate_dataset(
    "all_matches",
    match_cluster_dataset_all_df,
)

display(global_metrics_df.sort_values("balanced_accuracy", ascending=False))

## 12. Analyse Même Bucket UEFA

Second modèle : uniquement les matchs où les deux équipes sont dans le même intervalle de 20 places au ranking UEFA.

In [ ]:
same_bucket_metrics_df, same_bucket_confusion_df = evaluate_dataset(
    "same_rank_bucket",
    match_cluster_dataset_same_rank_bucket_df,
)

display(same_bucket_metrics_df.sort_values("balanced_accuracy", ascending=False))

## 13. Conclusion Et Exports Modèles

On compare les deux analyses avec les mêmes métriques et on sauvegarde les résultats.

In [ ]:
match_prediction_metrics_df = pd.concat(
    [global_metrics_df, same_bucket_metrics_df],
    ignore_index=True,
).sort_values(["dataset", "balanced_accuracy"], ascending=[True, False])

match_prediction_confusion_matrices_df = pd.concat(
    [global_confusion_df, same_bucket_confusion_df],
    ignore_index=True,
)

match_prediction_metrics_df.to_csv(OUTPUT_DIR / "match_prediction_metrics.csv", index=False)
match_prediction_confusion_matrices_df.to_csv(
    OUTPUT_DIR / "match_prediction_confusion_matrices.csv",
    index=False,
)

print("Métriques sauvegardées:", OUTPUT_DIR / "match_prediction_metrics.csv")
display(match_prediction_metrics_df)

tldr_rows = []
for dataset_name, dataset_metrics_df in match_prediction_metrics_df.groupby("dataset"):
    best_row = dataset_metrics_df.sort_values("balanced_accuracy", ascending=False).iloc[0]
    tldr_rows.append({
        "dataset": dataset_name,
        "best_model": best_row["model"],
        "balanced_accuracy": round(best_row["balanced_accuracy"], 3),
        "macro_f1": round(best_row["macro_f1"], 3),
        "test_rows": int(best_row["test_rows"]),
    })

tldr_df = pd.DataFrame(tldr_rows)
print("Résumé court")
display(tldr_df)